In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

results_dir = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results/ldbc_snb_sf0_1_full_fiben_format_clean"
)

agg = pd.read_csv(results_dir / "benchmark_aggregate_results.csv")

agg["official_id"] = agg["query_name"].str.extract(r"^(IC\d+|IS\d+|INS\d+)")

def query_group(oid):
    if oid.startswith("IC"):
        return "complex_read"
    if oid.startswith("IS"):
        return "short_read"
    if oid.startswith("INS"):
        return "insert"
    return "other"

agg["query_group"] = agg["official_id"].apply(query_group)

hot = agg[agg["run_phase"] == "hot"].copy()

rows = []

for query_name, grp in hot.groupby("query_name"):
    best_all = grp.loc[grp["p95_latency_ms"].idxmin()]

    activated = grp[grp["final_benchmark_group"] != "control"]
    primary = grp[grp["final_benchmark_group"] == "primary"]

    best_activated = activated.loc[activated["p95_latency_ms"].idxmin()]

    if len(primary) > 0:
        best_primary = primary.loc[primary["p95_latency_ms"].idxmin()]
        primary_regret = (
            best_primary["p95_latency_ms"] - best_all["p95_latency_ms"]
        ) / best_all["p95_latency_ms"]
    else:
        best_primary = None
        primary_regret = np.nan

    n_C = 10
    n_A = activated["candidate_id"].nunique()
    dsr = 1 - (n_A / n_C)

    rows.append({
        "official_id": best_all["official_id"],
        "query_name": query_name,
        "n_tested_configs": grp["candidate_id"].nunique(),
        "n_activated_configs": n_A,
        "DSR": dsr,

        "best_config": best_all["g_class"],
        "best_group": best_all["final_benchmark_group"],
        "best_design_pattern": best_all["design_pattern"],
        "best_p95_ms": best_all["p95_latency_ms"],

        "top1_preserved_by_activated": (
            best_activated["candidate_id"] == best_all["candidate_id"]
        ),
        "activated_regret": (
            best_activated["p95_latency_ms"] - best_all["p95_latency_ms"]
        ) / best_all["p95_latency_ms"],

        "best_primary_config": None if best_primary is None else best_primary["g_class"],
        "best_primary_p95_ms": None if best_primary is None else best_primary["p95_latency_ms"],
        "primary_regret": primary_regret,
    })

analysis_df = pd.DataFrame(rows).sort_values("official_id")

display(analysis_df)

print("Average DSR:", analysis_df["DSR"].mean())
print("Top-1 preservation activated:", analysis_df["top1_preserved_by_activated"].mean())
print("Mean activated regret:", analysis_df["activated_regret"].mean())
print("Mean primary regret:", analysis_df["primary_regret"].dropna().mean())

display(
    analysis_df[
        analysis_df["best_group"] == "secondary_affected"
    ][
        [
            "official_id",
            "query_name",
            "best_config",
            "best_design_pattern",
            "best_p95_ms",
            "best_primary_config",
            "best_primary_p95_ms",
            "primary_regret",
        ]
    ]
)

,official_id,query_name,n_tested_configs,n_activated_configs,DSR,best_config,best_group,best_design_pattern,best_p95_ms,top1_preserved_by_activated,activated_regret,best_primary_config,best_primary_p95_ms,primary_regret
0,IC1,IC1_TransitiveFriendsWithName,2,2,0.8,G3,primary,root_with_references_or_summaries,131.361695,True,0.0,G3,131.361695,0.000000
1,IC2,IC2_RecentMessagesByFriends,2,2,0.8,G3,primary,root_with_references_or_summaries,28.638829,True,0.0,G3,28.638829,0.000000
2,IC3,IC3_FriendsAndFriendsOfFriendsInCountries,4,4,0.6,G0,primary,root_with_references,164.711398,True,0.0,G0,164.711398,0.000000
3,IC4,IC4_NewTopics,2,2,0.8,G3,primary,root_with_references_or_summaries,44.056135,True,0.0,G3,44.056135,0.000000
4,IC5,IC5_NewGroups,6,6,0.4,G7,secondary_affected,containment_baseline,135.322968,True,0.0,G0,158.656699,0.172430
5,IC6,IC6_TagCoOccurrence,2,2,0.8,G3,primary,root_with_references_or_summaries,198.168794,True,0.0,G3,198.168794,0.000000
6,IC7,IC7_RecentLikers,4,4,0.6,G4,secondary_affected,explicit_edge_collection,7.464429,True,0.0,G0,10.145230,0.359143
7,INS1,INS1_AddPerson,2,2,0.8,G3,primary,root_with_references_or_summaries,1.184437,True,0.0,G3,1.184437,0.000000
8,INS2,INS2_AddLikeToPost,2,2,0.8,G4,primary,explicit_edge_collection,1.049605,True,0.0,G4,1.049605,0.000000
9,INS3,INS3_AddLikeToComment,2,2,0.8,G4,primary,explicit_edge_collection,1.222271,True,0.0,G4,1.222271,0.000000


Average DSR: 0.7136363636363637
Top-1 preservation activated: 1.0
Mean activated regret: 0.0
Mean primary regret: 0.02531301731556674


,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret
4,IC5,IC5_NewGroups,G7,containment_baseline,135.322968,G0,158.656699,0.172430
6,IC7,IC7_RecentLikers,G4,explicit_edge_collection,7.464429,G0,10.145230,0.359143
16,IS2,IS2_RecentMessagesOfPerson,G6,referenced_or_reverse_indexed_edges,2.255789,None,NaN,NaN
